# as-strided-windowing composite — cx8: pad first, then build the as_strided window view

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `conv-padding-zero`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "as-strided-windowing"
DD_ATOM_IDS = ["as-strided-windowing", "conv-padding-zero"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "CNN: Conv zero padding"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `conv1d_minimal` with padding splits the work into two zero-copy-friendly stages:
1. **Pad** the input with zeros on both ends (`conv-padding-zero`). This is a materializing op — it allocates a new tensor `x_pad` of shape `(B, IC, W + 2*P)`.
2. **Window via as_strided** (`as-strided-windowing`). Read `x_pad.stride()` and build a `(B, IC, OW, K)` view onto the PADDED tensor. No copy here — the view aliases `x_pad`'s storage.

**Why ordering matters.** You must read `x_pad.stride()`, NOT `x.stride()`. After padding, the storage and strides have changed (you're looking at a new contiguous tensor sized `W + 2*P`). Hard-coding `x.stride()` is one of the canonical ARENA bugs — your windows then read the wrong memory cells.

**Anatomy.**
- `x_pad = x.new_zeros(B, IC, W + 2*P); x_pad[..., P:P+W] = x` — atom B.
- `s_b, s_ic, s_w = x_pad.stride()` — read the PADDED tensor's strides.
- `OW = W + 2*P - K + 1` — stride-1 output length on the padded input.
- `win = x_pad.as_strided(size=(B, IC, OW, K), stride=(s_b, s_ic, s_w, s_w))` — atom A.

**Result.** `win` is a stride-1 windowed view of the padded input. Contract against a `(OC, IC, K)` kernel via `einops.einsum` to get the conv output.

### Composite Exercise — pad first, then build the as_strided window view

**Atoms exercised together**: `as-strided-windowing`, `conv-padding-zero`

Implement `cx8_padded_windows(x, K, P)`.

- `x`: float tensor of shape `(B, IC, W)`.
- `K`: kernel width.
- `P`: padding amount on each side (>= 0).

Return `(x_padded, win)`:
- `x_padded`: shape `(B, IC, W + 2*P)`. Standard zero-pad.
- `win`: shape `(B, IC, OW, K)` where `OW = W + 2*P - K + 1` — a stride-1 windowed view onto `x_padded`. Must share storage with `x_padded` (no copy).

1. **Pad** — use `x.new_zeros(...)` + slice assignment.
2. **Read PADDED strides** — `s_b, s_ic, s_w = x_padded.stride()`. Do NOT use `x.stride()` — after padding the strides are different.
3. **As-strided window** — `x_padded.as_strided(size=(B, IC, OW, K), stride=(s_b, s_ic, s_w, s_w))`. The trailing `(s_w, s_w)` pair is the stride-1 windowing pattern.

The test verifies the view is correct AND that `win.data_ptr() == x_padded.data_ptr()` (no copy), AND cross-checks against `F.conv1d(x, weight, padding=P)` after einsum-ing the windows against a random kernel.

In [ ]:
def cx8_padded_windows(x, K, P):
    B, IC, W = x.shape
    # Atom B (conv-padding-zero): zero-pad the input.
    x_pad = x.new_zeros(B, IC, W + 2 * P)
    x_pad[..., P : P + W] = x
    # Atom A (as-strided-windowing): read strides FROM THE PADDED TENSOR.
    s_b, s_ic, s_w = x_pad.stride()
    OW = W + 2 * P - K + 1
    win = x_pad.as_strided(
        size=(B, IC, OW, K),
        stride=(s_b, s_ic, s_w, s_w),
    )
    return x_pad, win


<details><summary>Show solution — cx8</summary>

```python
def cx8_padded_windows(x, K, P):
    B, IC, W = x.shape
    # Atom B (conv-padding-zero): zero-pad the input.
    x_pad = x.new_zeros(B, IC, W + 2 * P)
    x_pad[..., P : P + W] = x
    # Atom A (as-strided-windowing): read strides FROM THE PADDED TENSOR.
    s_b, s_ic, s_w = x_pad.stride()
    OW = W + 2 * P - K + 1
    win = x_pad.as_strided(
        size=(B, IC, OW, K),
        stride=(s_b, s_ic, s_w, s_w),
    )
    return x_pad, win
```

The critical move is reading `x_pad.stride()` — NOT `x.stride()`. After padding, you're looking at a new tensor with new (contiguous) strides. Hard-coding the pre-pad strides is the canonical ARENA bug: the windows would read garbage past the un-padded end of `x`. The stride-1 windowing pattern `(s_w, s_w)` for the trailing `(OW, K)` axes means 'advance by one element of the padded W axis to move between windows, and the same to walk within a window' — so adjacent windows overlap by `K - 1`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["PyTorch: as_strided windowing", "CNN: Conv zero padding"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()